# 03 — First features

Computes pre-fight rolling features on top of the de-biased fight table from
notebook 02, closing leakage traps #2 and #3 from `ROADMAP.md`: the raw
per-fight stats (strikes, takedowns, control time...) are outcomes of the
fight they're attached to, so they can't be used directly. This notebook
rolls them up per fighter, using only fights strictly before the one being
predicted, and attaches the static (time-invariant) attributes — age,
height, reach, stance — on top.

Weight is deliberately excluded: it changes across a fighter's career as
they move divisions, so a single joined value wouldn't be safe the way
reach/stance are — `weight_class` on the fight itself is the correct source
if weight is ever needed.

**Day 3 changes.**

*All remaining raw stats rolled* — including the strike-location breakdown
(head/body/leg, distance/clinch/ground) and `reversals`. Note these are exact
decompositions of `sig_strikes` (verified: it equals head+body+leg and
distance+clinch+ground on 100% of rows), so expect collinearity.

*Attempts + accuracy instead of landed + attempted* — since
`landed = attempts × accuracy`, that's a reparameterisation rather than a
loss, but it separates "how much they throw" from "how well it lands".
Accuracy is built in both directions: the fighter's own, and how often
opponents land what they throw *at* them.

*Volume normalised by fight length, both directions* — a raw count conflates
skill with fight length. Attempt counts, knockdowns, submissions and
reversals become per-minute rates; control time becomes a **share of the
bout**, since it already shares units with fight duration. This applies to
both what a fighter does (offence) and what happens to them (defence).
`total_fight_time` is rolled too, giving each fighter's average bout length
— useful on its own as a finisher/distance-goer signal, and the denominator
these rates need.

*Reach-to-height ratio ("ape index")* — `reach_cm − height_cm`. A distinct
scouting concept from either alone: two fighters can share identical reach
but very different frames depending on height.

*Stance matchup* — `same_stance`, a single symmetric boolean rather than a
diff (stance is categorical, and "is this an open matchup" is the same fact
regardless of which fighter is labelled A or B). A directional flag assuming
southpaw or orthodox has the advantage was deliberately not built — that's a
hypothesis for the model to test, not something to hard-code.

*Prior title and 5-round fight counts* — cumulative, the same pattern as
`wins`/`losses`. Title status comes from `weight_class`; 5-round status adds
any fight (title or not) that ran past 900 seconds, since a 3-round fight
cannot physically exceed that — this is a lower bound, since a 5-round fight
that ends by an early finish is indistinguishable from a 3-round fight
ending the same way.

As with notebook 02, the actual transform lives in
`src/ufc_predictor/features/` (`rolling.py`, `static.py`) — this notebook
calls it and checks the result.

In [ ]:
from ufc_predictor.data.io import load_processed, load_fighters, save_processed
from ufc_predictor.features.rolling import attach_prior_features
from ufc_predictor.features.static import attach_static_features

fights = load_processed("fights")
fighters = load_fighters()

featured = attach_prior_features(fights)
featured = attach_static_features(featured, fighters)
featured.shape

## Sanity checks

In [ ]:
# no fights lost or duplicated by the merge
assert len(featured) == len(fights)

# a genuine debut fighter has no rolled-up history to draw on
assert featured["fighter_a_prior_is_debut"].sum() > 0
debut_rows = featured[featured["fighter_a_prior_is_debut"]]
assert debut_rows["fighter_a_prior_sig_strike_atts_per_min_career_avg"].isna().all()
assert debut_rows["fighter_a_prior_sig_strikes_accuracy_career_avg"].isna().all()
assert debut_rows["fighter_a_prior_total_fight_time_career_avg"].isna().all()

any_debut_rate = (
    featured["fighter_a_prior_is_debut"] | featured["fighter_b_prior_is_debut"]
).mean()
print(f"fights involving at least one debutant: {any_debut_rate:.1%}")

In [ ]:
# leakage spot-check: Jim Miller's 3rd UFC fight (2009-03-07) should show a
# prior record of 2-0, not his eventual career record — this is the exact
# check that verified the module against a hand-computed example.
#
# he lands in the fighter_a slot for some fights and fighter_b for others
# (that's the point of the de-biasing in notebook 02), so both sides need
# checking, not just fighter_a.
jim_miller_id = 2500
mask = (featured["fighter_a_id"] == jim_miller_id) | (featured["fighter_b_id"] == jim_miller_id)
his_fights = featured[mask].sort_values("event_date")
his_3rd_fight = his_fights.iloc[2]
side = "fighter_a" if his_3rd_fight["fighter_a_id"] == jim_miller_id else "fighter_b"

prior_wins = his_3rd_fight[f"{side}_prior_wins"]
prior_losses = his_3rd_fight[f"{side}_prior_losses"]
print("date:", his_3rd_fight["event_date"].date())
print("prior_wins:", prior_wins)
print("prior_losses:", prior_losses)
assert prior_wins == 2
assert prior_losses == 0

In [ ]:
# static feature sanity checks: row count preserved, diffs are simple
# subtraction, missingness matches what we already knew about the raw
# fighter table (reach ~45%, stance ~19%) rather than something worse
assert len(featured) == len(fights)

diff_pairs = {
    "diff_static_age": ("fighter_a_age_during", "fighter_b_age_during"),
    "diff_static_height_cm": ("fighter_a_height_cm", "fighter_b_height_cm"),
    "diff_static_reach_cm": ("fighter_a_reach_cm", "fighter_b_reach_cm"),
}
for diff_col, (a_col, b_col) in diff_pairs.items():
    assert featured[diff_col].equals(featured[a_col] - featured[b_col])

print("reach missing:", f"{featured['fighter_a_reach_cm'].isna().mean():.1%}")
print("stance missing:", f"{featured['fighter_a_stance'].isna().mean():.1%}")
print("age missing:", f"{featured['fighter_a_age_during'].isna().mean():.1%}")

In [ ]:
# accuracy sanity checks: ratios must sit in [0, 1], and the defensive
# direction must carry information the raw absorbed volume doesn't — that
# near-zero correlation is the whole reason it was added.
#
# note the bound applies to the per-fighter columns only: diff_* columns are
# differences of two accuracies, so they legitimately span [-1, 1].
per_fighter_acc = [
    c for c in featured.columns
    if "_accuracy_" in c and c.startswith(("fighter_a_", "fighter_b_"))
]
print(f"{len(per_fighter_acc)} per-fighter accuracy columns")

for col in per_fighter_acc:
    vals = featured[col].dropna()
    assert vals.between(0, 1).all(), f"{col} outside [0,1]: {vals.min()}–{vals.max()}"

# control time absorbed is a share of the bout, so it cannot exceed 1.0.
# it did before the denominator was fixed to cover only the bouts that
# actually recorded control time — see _absorbed_rate_features.
for col in [c for c in featured.columns
            if "_frac_" in c and c.startswith(("fighter_a_", "fighter_b_"))]:
    vals = featured[col].dropna()
    assert vals.between(0, 1).all(), f"{col} outside [0,1]: {vals.min()}–{vals.max()}"

both = featured[[
    "fighter_a_prior_sig_strike_atts_absorbed_per_min_career_avg",
    "fighter_a_prior_sig_strikes_absorbed_accuracy_career_avg",
]].dropna()
corr = both.corr().iloc[0, 1]
print(f"corr(absorbed volume/min, opponent accuracy against fighter): {corr:.3f}")

print("\nopponent accuracy against fighter (how hittable they are):")
print(featured["fighter_a_prior_sig_strikes_absorbed_accuracy_career_avg"].describe()[
    ["mean", "std", "min", "max"]
].round(3).to_string())

print("\nshare of bout spent controlled by the opponent:")
print(featured["fighter_a_prior_ctrl_time_absorbed_frac_career_avg"].describe()[
    ["mean", "std", "min", "max"]
].round(3).to_string())

In [ ]:
# item 1: ape index. real spread, and now a feature rather than left implicit
# in reach_cm and height_cm separately.
ape = featured["fighter_a_ape_index_cm"].dropna()
print(f"ape index (reach - height): mean {ape.mean():.1f}cm, std {ape.std():.1f}, "
      f"range {ape.min():.1f} to {ape.max():.1f}")

# item 3: stance matchup. a symmetric fact, not diffed.
print(f"\nsame_stance dtype: {featured['same_stance'].dtype}")
print(featured["same_stance"].value_counts(dropna=False))

# item 4: title / five-round counts are cumulative and monotonic per fighter,
# same shape as wins/losses. sanity-check against a known multi-title veteran.
top = featured.loc[featured["fighter_a_prior_title_fights"].idxmax()]
print(f"\nmost prior UFC title fights (fighter_a slot): {top['fighter_a_prior_title_fights']:.0f} "
      f"— {top['fighter_a_name']} on {top['event_date'].date()}")

assert featured["fighter_a_prior_five_round_fights"].ge(
    featured["fighter_a_prior_title_fights"]
).all(), "every title fight is 5-round, so five_round_fights can't be less than title_fights"

## A look at the engineered features

In [ ]:
diff_cols = [c for c in featured.columns if c.startswith("diff_prior_") or c.startswith("diff_static_")]
print(f"{len(diff_cols)} diff features")
featured[diff_cols].describe().T[["mean", "std", "min", "max"]]

## Save

In [ ]:
save_processed(featured, "fights_featured")
featured.head()

## Visualise the dataset

The full table is far too wide to read directly. A curated view of the most
recent fights: who fought, the label, and the headline engineered features —
striking volume, accuracy in both directions, how much of the bout they
spend controlled, and typical bout length.

In [ ]:
readable_cols = [
    "event_date", "weight_class",
    "fighter_a_name", "fighter_b_name", "fighter_a_won",
    "diff_prior_win_rate", "diff_prior_streak",
    "diff_prior_sig_strike_atts_per_min_career_avg",       # striking volume, per minute
    "diff_prior_sig_strikes_accuracy_career_avg",          # striking accuracy (offence)
    "diff_prior_sig_strikes_absorbed_accuracy_career_avg", # how hittable they are
    "diff_prior_ctrl_time_frac_career_avg",                # share of bout THEY controlled
    "diff_prior_title_fights",                             # title-fight experience
    "diff_static_age", "diff_static_reach_cm", "diff_static_ape_index_cm",
    "same_stance",
]

view = featured[readable_cols].sort_values("event_date", ascending=False)
view.head(20)

In [ ]:
# load the saved wide table back in a fresh session, for inspection
from ufc_predictor.data.io import load_processed

df = load_processed("fights_featured")
df.shape